In [ ]:
#数据源 2026.6.10实时提取的这个平台上的所有德国 语言英语/德语的零工工作

import json
import pandas as pd
import numpy as np

# Fair Pay 参考：德国法定最低时薪（2026 前后约 €12.82–13，可按需要改）
M_DE_EUR = 13.0
EUR_TO_USD = 1.15564  # JSON 里 EUR 的 exchange_rate，用于统一换算

def to_eur(amount, exchange_rate):
    """将原货币金额换算为 EUR（基于 Freelancer 的 exchange_rate 字段）"""
    if amount is None or (isinstance(amount, float) and np.isnan(amount)):
        return np.nan
    amount_usd = amount * exchange_rate
    return amount_usd / EUR_TO_USD

with open("data/freelance.json", encoding="utf-8") as f:
    projects = json.load(f)

rows = []
for p in projects:
    budget = p.get("budget", {}) or {}
    currency = p.get("currency", {}) or {}
    bid_stats = p.get("bid_stats", {}) or {}

    budget_min = budget.get("minimum")
    budget_max = budget.get("maximum")
    exchange_rate = currency.get("exchange_rate", np.nan)
    bid_avg = bid_stats.get("bid_avg")

    # --- 5 个衍生变量 ---
    # 1) budget_mid
    if budget_min is not None and budget_max is not None:
        budget_mid = (budget_min + budget_max) / 2
    elif budget_min is not None:
        budget_mid = budget_min
    elif budget_max is not None:
        budget_mid = budget_max
    else:
        budget_mid = np.nan

    # 2) budget_range
    budget_range = (
        budget_max - budget_min
        if budget_min is not None and budget_max is not None
        else np.nan
    )

    # 3) bid_avg_eur（薪酬代理变量，统一成 EUR）
    bid_avg_eur = to_eur(bid_avg, exchange_rate)

    # 4) bid_vs_budget（<1 表示竞标均价低于挂牌中位）
    budget_mid_eur = to_eur(budget_mid, exchange_rate)
    bid_vs_budget = (
        bid_avg_eur / budget_mid_eur
        if pd.notna(bid_avg_eur) and pd.notna(budget_mid_eur) and budget_mid_eur != 0
        else np.nan
    )

    # 5) below_min_wage（Fair Pay 1.2 代理：时薪项目是否低于德国最低工资）
    budget_min_eur = to_eur(budget_min, exchange_rate)
    budget_max_eur = to_eur(budget_max, exchange_rate)
    if p.get("type") == "hourly":
        below_min_wage = (
            (pd.notna(budget_max_eur) and budget_max_eur < M_DE_EUR)
            or (pd.notna(bid_avg_eur) and bid_avg_eur < M_DE_EUR)
        )
    else:
        below_min_wage = np.nan  # fixed 项目需另做「假设工时」分析

    rows.append({
        # 必取字段
        "id": p.get("id"),
        "title": p.get("title"),
        "type": p.get("type"),
        "budget_min": budget_min,
        "budget_max": budget_max,
        "currency_code": currency.get("code"),
        "exchange_rate": exchange_rate,
        "bid_avg": bid_avg,
        "bid_count": bid_stats.get("bid_count"),
        "language": p.get("language"),
        "status": p.get("status"),
        # Fair Pay 辅助（换算后更直观）
        "budget_min_eur": budget_min_eur,
        "budget_max_eur": budget_max_eur,
        "budget_mid_eur": budget_mid_eur,
        # 5 个衍生变量
        "budget_mid": budget_mid,
        "budget_range": budget_range,
        "bid_avg_eur": bid_avg_eur,
        "bid_vs_budget": bid_vs_budget,
        "below_min_wage": below_min_wage,
    })

df = pd.DataFrame(rows)
df.head()

,id,title,type,budget_min,budget_max,currency_code,exchange_rate,bid_avg,bid_count,language,status,budget_min_eur,budget_max_eur,budget_mid_eur,budget_mid,budget_range,bid_avg_eur,bid_vs_budget,below_min_wage
0,40505114,WebScrapper for a real estate information -- 2,fixed,30,250.0,EUR,1.15564,143.038095,105.0,en,active,30.000000,250.000000,140.000000,140.0,220.0,143.038095,1.021701,NaN
1,40505063,German Sales Appointment Setter,fixed,1500,3000.0,EUR,1.15564,2012.500000,4.0,en,active,1500.000000,3000.000000,2250.000000,2250.0,1500.0,2012.500000,0.894444,NaN
2,40504861,SolidWorks Winkelanpassung mit STEP-Ausgabe,hourly,12,18.0,EUR,1.15564,15.652174,23.0,de,active,12.000000,18.000000,15.000000,15.0,6.0,15.652174,1.043478,False
3,40504835,Request for SAP CPI / SAP Integration Develope...,fixed,5000,10000.0,EUR,1.15564,6950.652174,23.0,en,active,5000.000000,10000.000000,7500.000000,7500.0,5000.0,6950.652174,0.926754,NaN
4,40504799,E-Commerce Subscription Website Developer – Su...,fixed,30,250.0,USD,1.00000,159.137029,239.0,en,active,25.959641,216.330345,121.144993,140.0,220.0,137.704674,1.136693,NaN


In [4]:
hourly = df[df["type"] == "hourly"].copy()

print("=== Fair Pay 1.2 探索（hourly, N={}) ===".format(len(hourly)))
print("预算上限 < €{}: ".format(M_DE_EUR),
      (hourly["budget_max_eur"] < M_DE_EUR).sum())
print("bid_avg < €{}: ".format(M_DE_EUR),
      (hourly["bid_avg_eur"] < M_DE_EUR).sum())
print("below_min_wage = True: ",
      hourly["below_min_wage"].sum())

hourly[["title", "budget_min_eur", "budget_max_eur", "bid_avg_eur", "below_min_wage"]]

=== Fair Pay 1.2 探索（hourly, N=14) ===
预算上限 < €13.0:  6
bid_avg < €13.0:  6
below_min_wage = True:  6


,title,budget_min_eur,budget_max_eur,bid_avg_eur,below_min_wage
2,SolidWorks Winkelanpassung mit STEP-Ausgabe,12.000000,18.000000,15.652174,False
5,Temporary Instagram Outreach Assistant,2.000000,6.000000,4.023649,True
12,Virtual Assistant Support Needed in Bulgaria,6.000000,12.000000,9.000000,True
13,Professional 3D Website Visuals,15.000000,60.000000,30.571429,False
16,Temporary Finance & Admin Assistant Needed in ...,18.000000,36.000000,23.588235,False
19,AI Interview Practice and Real Time Coaching W...,21.633035,43.266069,34.912575,False
20,Personal Biography Ghostwriting,2.000000,6.000000,3.977273,True
23,Veranstaltungstechniker für Konzert am 14. Jun...,12.000000,18.000000,NaN,False
25,**Siemens TIA Portal Commissioning Engineer (F...,40.000000,60.000000,49.111111,False
26,Projection Mapping Specialist Needed –Immersiv...,36.000000,NaN,NaN,False
